# 2.7 — Modèles non paramétriques : SVM et k plus proches voisins

**Navigation** : [<< 2.6-Clustering-KMeans-PCA](2.6-Clustering-KMeans-PCA.ipynb) | Version Python : [2.7-Modeles-Non-Parametriques](2.7-Modeles-Non-Parametriques.ipynb) | Compléments : [2.7b SMO from scratch](2.7b-SMO-From-Scratch.ipynb) · [2.7c SVM SOTA](2.7c-SVM-SOTA-Comparison.ipynb) | [Index](../README.md)

**Kernel** : Rust (evcxr)

> **Jumeau Rust.** Ce carnet est le portage du carnet Python [2.7-Modeles-Non-Parametriques](2.7-Modeles-Non-Parametriques.ipynb) : même fil pédagogique, mêmes instances numériques (300 points, même bruit, mêmes valeurs de `C` et de `k`). Trois différences de forme, à connaître avant de lire :
>
> - la bibliothèque est **`smartcore` 0.7.0** à la place de scikit-learn, et les générateurs de jeux de données jouets (`make_moons`, `make_circles`) y sont réécrits à la main dans les cellules qui les utilisent ;
> - les figures deviennent des **cartes ASCII** de 64 colonnes × 24 lignes : chaque case du plan porte `.` si le modèle y prédit la classe 0, `o` pour la classe 1 (`*` marque, sur la carte des points, une case où tombent des points des deux classes) ; la première ligne affichée est le haut du plan ;
> - les lignes affichées par le code sont reprises **sans accents, à l'identique de la source** : ce sont des témoins de comparaison entre les deux carnets. Le tirage aléatoire vient d'un générateur maison, les nuages de points diffèrent donc un peu de la source, et les précisions avec eux.

## Introduction

Les modèles des notebooks précédents sont **paramétriques** : la régression linéaire/logistique (2.3) apprend un vecteur de coefficients de taille fixe, les arbres et forêts (2.4) apprennent un ensemble fini de règles de division. Une fois entraînés, **la taille du modèle ne dépend plus du nombre de données**. Les modèles **non paramétriques** que nous étudions ici renversent cette hypothèse : leur complexité **croît avec les données**. Un SVM garde en mémoire certains points d'entraînement (les *vecteurs supports*), un k-NN garde... **tous** les points. Aucune « formule » n'est apprise au sens classique ; la prédiction repose sur la comparaison aux données stockées.

Ce notebook présente les deux familles non paramétriques supervisées canoniques :

- les **Support Vector Machines (SVM)**, qui plongent les données dans un espace de dimension supérieure via un **noyau** pour y tracer un hyperplan séparateur à marge maximale ;
- les **k plus proches voisins (k-NN)**, qui classent un point par vote majoritaire parmi ses `k` voisins les plus proches.

Le **concept-phare** est le **kernel trick rendu visible** : sur un jeu de données **non séparable linéairement** (deux demi-lunes imbriquées), un SVM à noyau linéaire échoue tandis qu'un SVM à noyau RBF trace une frontière courbe qui épouse les classes. C'est la démonstration directe qu'un séparateur linéaire dans un espace transformé équivaut à une frontière **non linéaire** dans l'espace d'origine.

> **Note.** Le clustering non supervisé (KMeans, ACP) est traité dans le notebook [2.6](2.6-Clustering-KMeans-PCA.ipynb). Ici, les étiquettes `y` sont **connues** : nous sommes en classification supervisée.

### Objectifs d'apprentissage

À la fin de ce notebook, vous saurez :

1. Ajuster un SVM à noyaux linéaire et RBF et comparer leurs frontières de décision sur des données non séparables linéairement.
2. Comprendre en quoi la frontière d'un SVM ne dépend **que** des **vecteurs supports** — que la version Rust de `SVC` ne donne pas à lire (aucun accesseur public dans smartcore 0.7.0) : le jumeau compte à la place les points mal classés à l'entraînement.
3. Régler le paramètre `C` (marge) dans l'exercice 1 ; connaître le rôle du paramètre `gamma` du noyau RBF — avec `C`, l'une des deux molettes de flexibilité du SVM, laissé à sa valeur `"scale"` dans ce carnet (calculée explicitement : smartcore ne la fournit pas), et exploré par `GridSearchCV` dans [2.7c](2.7c-SVM-SOTA-Comparison.ipynb).
4. Appliquer le k-NN, mesurer l'effet du nombre de voisins `k` et démontrer pourquoi la **standardisation** des variables est indispensable.
5. Construire une pipeline complète (standardisation + classifieur) sur un jeu de données réel (cancer du sein : le jeu `load_breast_cancer` de la source, lu ici dans `smartcore::dataset::breast_cancer`).

### Prérequis

- Notebook 2.1 (workflow ML, surapprentissage).
- Notebook 2.3 (modèles linéaires : frontière comme hyperplan).
- Notebook 2.4 (frontière de décision, réduction de variance).

> **Référence.** Cortes, C. & Vapnik, V. (1995), *Support-Vector Networks*, Machine Learning 20(3):273-297. Article fondateur des SVM pour la classification : on cherche l'hyperplan qui maximise la **marge** (la distance minimale aux points les plus proches), éventuellement dans un espace transformé par un noyau.

In [2]:
// Configuration du jumeau Rust du notebook 2.7 : smartcore 0.7.0, feature datasets.
// La dependance se declare en tete de carnet, une seule fois : la crate n'est
// compilee qu'a la premiere execution.
:dep smartcore = { version = "0.7.0", features = ["datasets"] }

println!("Configuration OK : 2.7 - Modeles non parametriques (SVM et k-NN)");

Configuration OK : 2.7 - Modeles non parametriques (SVM et k-NN)


## 1. Des données non séparables linéairement : les demi-lunes

Pour que le noyau RBF soit **nécessaire** (et non un caprice), il nous faut un jeu de données qu'**aucune droite** ne peut séparer correctement. `make_moons` génère deux demi-lunes imbriquées : c'est le cas canonique. Nous générons ce jeu puis le traçons. Tout au long du notebook, ces données serviront de banc d'essai pour comparer les noyaux et mesurer l'effet des hyperparamètres.

> **Côté Rust.** smartcore ne fournit pas de générateur de jeux de données jouets : `make_moons` est réécrit dans la cellule ci-dessous — deux demi-cercles de rayon 1, bruit gaussien d'écart 0.30, mélange des points et bruit pilotés par la graine 42. Le tracé est une carte ASCII des points d'entraînement.

In [3]:
// Jeu de donnees : deux demi-lunes imbriquees (non separables lineairement)
use smartcore::linalg::basic::matrix::DenseMatrix;
use smartcore::linalg::basic::arrays::Array;

// Bruit : melange de Fisher-Yates portant sur les points ET les etiquettes (meme
// permutation), puis bruit gaussien. Generateur xorshift64 : la graine suffit a
// reproduire le jeu de donnees.
fn bruit(x: &mut Vec<Vec<f64>>, y: &mut Vec<u32>, ecart: f64, graine: u64) {
    let mut etat = graine | 1;
    let mut uniform = || {
        etat ^= etat << 13;
        etat ^= etat >> 7;
        etat ^= etat << 17;
        (etat >> 11) as f64 / (1u64 << 53) as f64
    };
    let n = x.len();
    for i in (1..n).rev() {
        let j = (uniform() * (i + 1) as f64) as usize;
        x.swap(i, j);
        y.swap(i, j);
    }
    let mut gaussien = || {
        let u = uniform().max(1e-12);
        (-2.0 * u.ln()).sqrt() * (2.0 * std::f64::consts::PI * uniform()).cos()
    };
    for p in x.iter_mut() {
        p[0] += ecart * gaussien();
        p[1] += ecart * gaussien();
    }
}

// make_moons maison : deux demi-cercles de rayon 1 (exterieur en haut, interieur
// en bas), meme effectif par classe, puis bruit (ecart, graine).
fn make_moons(n: usize, ecart: f64, graine: u64) -> (DenseMatrix<f64>, Vec<u32>) {
    let n_ext = n / 2;
    let n_int = n - n_ext;
    let mut x: Vec<Vec<f64>> = Vec::with_capacity(n);
    let mut y: Vec<u32> = Vec::with_capacity(n);
    for i in 0..n_ext {
        let t = std::f64::consts::PI * i as f64 / (n_ext - 1) as f64;
        x.push(vec![t.cos(), t.sin()]);
        y.push(0);
    }
    for i in 0..n_int {
        let t = std::f64::consts::PI * i as f64 / (n_int - 1) as f64;
        x.push(vec![1.0 - t.cos(), 0.5 - t.sin()]);
        y.push(1);
    }
    bruit(&mut x, &mut y, ecart, graine);
    (DenseMatrix::from_2d_vec(&x).unwrap(), y)
}

// Bornes du plan : etendue des donnees, elargie de 0.5 de chaque cote (comme la source).
fn bornes(x: &DenseMatrix<f64>) -> (f64, f64, f64, f64) {
    let mut x0 = f64::INFINITY;
    let mut x1 = f64::NEG_INFINITY;
    let mut y0 = f64::INFINITY;
    let mut y1 = f64::NEG_INFINITY;
    for i in 0..x.shape().0 {
        let (px, py) = (*x.get((i, 0)), *x.get((i, 1)));
        x0 = x0.min(px);
        x1 = x1.max(px);
        y0 = y0.min(py);
        y1 = y1.max(py);
    }
    (x0 - 0.5, x1 + 0.5, y0 - 0.5, y1 + 0.5)
}

// Carte ASCII des points d'entrainement : '.' = classe 0, 'o' = classe 1,
// '*' = des points des deux classes dans la meme case. Premiere ligne = haut du plan.
fn carte_points(x: &DenseMatrix<f64>, y: &[u32], ncol: usize, nlig: usize) -> String {
    let (x0, x1, y0, y1) = bornes(x);
    let mut grille = vec![vec![' '; ncol]; nlig];
    for i in 0..x.shape().0 {
        let c = (((*x.get((i, 0)) - x0) / (x1 - x0)) * ncol as f64) as usize;
        let r = (((*x.get((i, 1)) - y0) / (y1 - y0)) * nlig as f64) as usize;
        let symbole = if y[i] == 0 { '.' } else { 'o' };
        grille[r][c] = if grille[r][c] != ' ' && grille[r][c] != symbole { '*' } else { symbole };
    }
    (0..nlig).rev().map(|r| grille[r].iter().collect::<String>()).collect::<Vec<String>>().join("\n")
}

let (x_moons, y_moons) = make_moons(300, 0.30, 42);

let mut classes: Vec<u32> = y_moons.clone();
classes.sort_unstable();
classes.dedup();
let classes_txt = classes.iter().map(|c| c.to_string()).collect::<Vec<String>>().join(" ");

println!("X_moons : ({}, {})   y_moons : ({},)", x_moons.shape().0, x_moons.shape().1, y_moons.len());
println!("Classes : [{}]   (0 et 1)", classes_txt);
println!();
println!("Carte : deux demi-lunes imbriquees (make_moons, noise=0.30)");
println!("{}", carte_points(&x_moons, &y_moons, 64, 24));
// Aucune droite ne separe proprement les deux demi-lunes : un modele
// strictement lineaire echouera, un modele capable de courber reussira.

X_moons : (300, 2)   y_moons : (300,)


Classes : [0 1]   (0 et 1)


Carte : deux demi-lunes imbriquees (make_moons, noise=0.30)


                     .                                          


                          ..                                    


                   .           .                                


... (11 outputs left out)

                           o  oo o o * oooooo   o               


                                o   o       o    o              


                                              o                 


                               o                                


## 2. SVM à noyau linéaire : la limite du séparateur droit

Un **Support Vector Machine** cherche l'hyperplan qui **maximise la marge** — la distance minimale entre la frontière et les points d'entraînement les plus proches (les **vecteurs supports**). Avec un **noyau linéaire**, la frontière est... une droite (en 2D). Sur les demi-lunes, ce séparateur ne peut qu'échouer : c'est précisément ce que nous voulons montrer, car il justifie l'introduction d'un noyau non linéaire à la section suivante.

> **Côté Rust.** La source définit une fonction utilitaire `frontiere_decision` (ré-ajustée sur les 2 variables pour la visualisation, comme dans le notebook 2.4). Le jumeau la remplace par `carte_decision` : le modèle est ajusté une seule fois sur les 2 variables, puis la grille de 64 colonnes × 24 lignes du plan lui est soumise case par case et la grille est imprimée. Les bornes du plan sont celles de la source : étendue des données, élargie de 0.5 de chaque côté.

> **Contraintes de smartcore.** `SVC::fit` emprunte ses paramètres : ils se lient à une variable **avant** l'appel. Le modèle ajusté emprunte aussi ses données, et le noyau Jupyter persiste les liaisons de haut niveau d'une cellule : ajustement et prédiction vivent donc dans un bloc, dont le résultat (accuracy, carte) sort sous forme de valeurs possédées.

In [4]:
// SVM a noyau lineaire sur les demi-lunes (C = 1)
use smartcore::linalg::basic::matrix::DenseMatrix;
use smartcore::linalg::basic::arrays::Array;
use smartcore::svm::{Kernels, svc::{SVC, SVCParameters}};

type P = SVCParameters<f64, u32, DenseMatrix<f64>, Vec<u32>>;

// Bruit : melange de Fisher-Yates portant sur les points ET les etiquettes (meme
// permutation), puis bruit gaussien. Generateur xorshift64 : la graine suffit a
// reproduire le jeu de donnees.
fn bruit(x: &mut Vec<Vec<f64>>, y: &mut Vec<u32>, ecart: f64, graine: u64) {
    let mut etat = graine | 1;
    let mut uniform = || {
        etat ^= etat << 13;
        etat ^= etat >> 7;
        etat ^= etat << 17;
        (etat >> 11) as f64 / (1u64 << 53) as f64
    };
    let n = x.len();
    for i in (1..n).rev() {
        let j = (uniform() * (i + 1) as f64) as usize;
        x.swap(i, j);
        y.swap(i, j);
    }
    let mut gaussien = || {
        let u = uniform().max(1e-12);
        (-2.0 * u.ln()).sqrt() * (2.0 * std::f64::consts::PI * uniform()).cos()
    };
    for p in x.iter_mut() {
        p[0] += ecart * gaussien();
        p[1] += ecart * gaussien();
    }
}

// make_moons maison : deux demi-cercles de rayon 1 (exterieur en haut, interieur
// en bas), meme effectif par classe, puis bruit (ecart, graine).
fn make_moons(n: usize, ecart: f64, graine: u64) -> (DenseMatrix<f64>, Vec<u32>) {
    let n_ext = n / 2;
    let n_int = n - n_ext;
    let mut x: Vec<Vec<f64>> = Vec::with_capacity(n);
    let mut y: Vec<u32> = Vec::with_capacity(n);
    for i in 0..n_ext {
        let t = std::f64::consts::PI * i as f64 / (n_ext - 1) as f64;
        x.push(vec![t.cos(), t.sin()]);
        y.push(0);
    }
    for i in 0..n_int {
        let t = std::f64::consts::PI * i as f64 / (n_int - 1) as f64;
        x.push(vec![1.0 - t.cos(), 0.5 - t.sin()]);
        y.push(1);
    }
    bruit(&mut x, &mut y, ecart, graine);
    (DenseMatrix::from_2d_vec(&x).unwrap(), y)
}

// Accuracy : fraction des etiquettes bien predites. SVC::predict rend un f64 par
// point (la classe 0.0 ou 1.0), d'ou la comparaison convertie.
fn precision_f64(y: &[u32], p: &[f64]) -> f64 {
    y.iter().zip(p).filter(|(a, b)| **a as f64 == **b).count() as f64 / y.len() as f64
}

// Bornes du plan : etendue des donnees, elargie de 0.5 de chaque cote (comme la source).
fn bornes(x: &DenseMatrix<f64>) -> (f64, f64, f64, f64) {
    let mut x0 = f64::INFINITY;
    let mut x1 = f64::NEG_INFINITY;
    let mut y0 = f64::INFINITY;
    let mut y1 = f64::NEG_INFINITY;
    for i in 0..x.shape().0 {
        let (px, py) = (*x.get((i, 0)), *x.get((i, 1)));
        x0 = x0.min(px);
        x1 = x1.max(px);
        y0 = y0.min(py);
        y1 = y1.max(py);
    }
    (x0 - 0.5, x1 + 0.5, y0 - 0.5, y1 + 0.5)
}

// Carte ASCII de la frontiere : la grille ncol x nlig du plan est classee par le
// modele deja ajuste ; '.' = classe 0, 'o' = classe 1. Premiere ligne = haut du plan.
fn carte_decision(pred: impl Fn(&DenseMatrix<f64>) -> Vec<u32>, x: &DenseMatrix<f64>, ncol: usize, nlig: usize) -> String {
    let (x0, x1, y0, y1) = bornes(x);
    let mut lignes = Vec::with_capacity(nlig);
    for r in (0..nlig).rev() {
        let yy = y0 + (y1 - y0) * (r as f64 + 0.5) / nlig as f64;
        let grille: Vec<Vec<f64>> = (0..ncol)
            .map(|c| vec![x0 + (x1 - x0) * (c as f64 + 0.5) / ncol as f64, yy])
            .collect();
        let classes = pred(&DenseMatrix::from_2d_vec(&grille).unwrap());
        lignes.push(classes.iter().map(|&c| if c == 0 { '.' } else { 'o' }).collect::<String>());
    }
    lignes.join("\n")
}

let (x_moons, y_moons) = make_moons(300, 0.30, 42);

// SVM a noyau lineaire. Les parametres se lient a une variable avant l'ajustement ;
// le modele ajuste emprunte ses donnees, il vit dans le bloc.
let params_lin = P::default().with_c(1.0).with_kernel(Kernels::linear()).with_seed(Some(42));
let (acc_lin, carte_lin) = {
    let svm_lin = SVC::fit(&x_moons, &y_moons, &params_lin).unwrap();
    let acc = precision_f64(&y_moons, &svm_lin.predict(&x_moons).unwrap());
    let carte = carte_decision(
        |grille| svm_lin.predict(grille).unwrap().into_iter().map(|v| v as u32).collect(),
        &x_moons,
        64,
        24,
    );
    (acc, carte)
};

println!("Carte : SVM lineaire (C=1.0) -- acc = {:.3}", acc_lin);
println!("{}", carte_lin);
println!();
println!("SVM lineaire -- accuracy sur les demi-lunes : {:.3}", acc_lin);
println!("La frontiere est une DROITE : elle coupe inevitablement les demi-lunes.");
println!("Un noyau non lineaire devient necessaire (section 3).");

Carte : SVM lineaire (C=1.0) -- acc = 0.860


................................................................


................................................................


................................................................


................................................................


................................................................


... (18 outputs left out)

oooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooo


SVM lineaire -- accuracy sur les demi-lunes : 0.860


La frontiere est une DROITE : elle coupe inevitablement les demi-lunes.


Un noyau non lineaire devient necessaire (section 3).


## 3. Le kernel trick : plonger dans un espace où tout devient linéaire

L'idée du **kernel trick** : si les données ne sont pas séparables dans l'espace d'origine, **transformons-les** par une application $\phi$ vers un espace de dimension supérieure où elles le deviennent, puis traçons-y l'hyperplan à marge maximale. Le tour de force est qu'on n'a **jamais besoin de calculer $\phi$ explicitement** : le **noyau** $K(x, x') = \langle \phi(x), \phi(x') \rangle$ donne directement le produit scalaire dans l'espace transformé. Le noyau **RBF** (gaussien) $K(x, x') = \exp(-\gamma \|x - x'\|^2)$ plonge les données dans un espace de dimension infinie : un hyperplan linéaire là-dedans correspond à une frontière **courbe** dans l'espace d'origine, qui épouse les classes.

Comparons côte à côte le noyau linéaire (échec) et le noyau RBF (succès) : la frontière du SVM ne dépend **que** des **vecteurs supports** — c'est le caractère non paramétrique incarné. La source entoure ces points en vert ; `SVC` de smartcore 0.7.0 n'en donne aucun accès, le jumeau compte donc à la place les points **mal classés à l'entraînement**.

> **Référence.** Cristianini, N. & Shawe-Taylor, J. (2000), *An Introduction to Support Vector Machines and Other Kernel-based Learning Methods*, Cambridge University Press. Présentation pédagogique du kernel trick : le noyau calcule un produit scalaire dans un espace transformé sans jamais expliciter la transformation $\phi$.

In [5]:
// SVM a noyau RBF sur les demi-lunes (C = 1, gamma "scale", graine 42)
use smartcore::linalg::basic::matrix::DenseMatrix;
use smartcore::linalg::basic::arrays::Array;
use smartcore::svm::{Kernels, svc::{SVC, SVCParameters}};

type P = SVCParameters<f64, u32, DenseMatrix<f64>, Vec<u32>>;

// Bruit : melange de Fisher-Yates portant sur les points ET les etiquettes (meme
// permutation), puis bruit gaussien. Generateur xorshift64 : la graine suffit a
// reproduire le jeu de donnees.
fn bruit(x: &mut Vec<Vec<f64>>, y: &mut Vec<u32>, ecart: f64, graine: u64) {
    let mut etat = graine | 1;
    let mut uniform = || {
        etat ^= etat << 13;
        etat ^= etat >> 7;
        etat ^= etat << 17;
        (etat >> 11) as f64 / (1u64 << 53) as f64
    };
    let n = x.len();
    for i in (1..n).rev() {
        let j = (uniform() * (i + 1) as f64) as usize;
        x.swap(i, j);
        y.swap(i, j);
    }
    let mut gaussien = || {
        let u = uniform().max(1e-12);
        (-2.0 * u.ln()).sqrt() * (2.0 * std::f64::consts::PI * uniform()).cos()
    };
    for p in x.iter_mut() {
        p[0] += ecart * gaussien();
        p[1] += ecart * gaussien();
    }
}

// make_moons maison : deux demi-cercles de rayon 1 (exterieur en haut, interieur
// en bas), meme effectif par classe, puis bruit (ecart, graine).
fn make_moons(n: usize, ecart: f64, graine: u64) -> (DenseMatrix<f64>, Vec<u32>) {
    let n_ext = n / 2;
    let n_int = n - n_ext;
    let mut x: Vec<Vec<f64>> = Vec::with_capacity(n);
    let mut y: Vec<u32> = Vec::with_capacity(n);
    for i in 0..n_ext {
        let t = std::f64::consts::PI * i as f64 / (n_ext - 1) as f64;
        x.push(vec![t.cos(), t.sin()]);
        y.push(0);
    }
    for i in 0..n_int {
        let t = std::f64::consts::PI * i as f64 / (n_int - 1) as f64;
        x.push(vec![1.0 - t.cos(), 0.5 - t.sin()]);
        y.push(1);
    }
    bruit(&mut x, &mut y, ecart, graine);
    (DenseMatrix::from_2d_vec(&x).unwrap(), y)
}

// gamma = "scale" de la source : 1 / (nombre de variables x variance de X), la
// variance etant populationnelle et porte sur toutes les valeurs de X.
// smartcore ne fournit pas cette valeur par defaut : elle se calcule.
fn gamma_scale(m: &DenseMatrix<f64>) -> f64 {
    let v: Vec<f64> = m.iter().cloned().collect();
    let n = v.len() as f64;
    let mu: f64 = v.iter().sum::<f64>() / n;
    let variance: f64 = v.iter().map(|z| (z - mu) * (z - mu)).sum::<f64>() / n;
    1.0 / (m.shape().1 as f64 * variance)
}

// Accuracy : fraction des etiquettes bien predites. SVC::predict rend un f64 par
// point (la classe 0.0 ou 1.0), d'ou la comparaison convertie.
fn precision_f64(y: &[u32], p: &[f64]) -> f64 {
    y.iter().zip(p).filter(|(a, b)| **a as f64 == **b).count() as f64 / y.len() as f64
}

// Bornes du plan : etendue des donnees, elargie de 0.5 de chaque cote (comme la source).
fn bornes(x: &DenseMatrix<f64>) -> (f64, f64, f64, f64) {
    let mut x0 = f64::INFINITY;
    let mut x1 = f64::NEG_INFINITY;
    let mut y0 = f64::INFINITY;
    let mut y1 = f64::NEG_INFINITY;
    for i in 0..x.shape().0 {
        let (px, py) = (*x.get((i, 0)), *x.get((i, 1)));
        x0 = x0.min(px);
        x1 = x1.max(px);
        y0 = y0.min(py);
        y1 = y1.max(py);
    }
    (x0 - 0.5, x1 + 0.5, y0 - 0.5, y1 + 0.5)
}

// Carte ASCII de la frontiere : la grille ncol x nlig du plan est classee par le
// modele deja ajuste ; '.' = classe 0, 'o' = classe 1. Premiere ligne = haut du plan.
fn carte_decision(pred: impl Fn(&DenseMatrix<f64>) -> Vec<u32>, x: &DenseMatrix<f64>, ncol: usize, nlig: usize) -> String {
    let (x0, x1, y0, y1) = bornes(x);
    let mut lignes = Vec::with_capacity(nlig);
    for r in (0..nlig).rev() {
        let yy = y0 + (y1 - y0) * (r as f64 + 0.5) / nlig as f64;
        let grille: Vec<Vec<f64>> = (0..ncol)
            .map(|c| vec![x0 + (x1 - x0) * (c as f64 + 0.5) / ncol as f64, yy])
            .collect();
        let classes = pred(&DenseMatrix::from_2d_vec(&grille).unwrap());
        lignes.push(classes.iter().map(|&c| if c == 0 { '.' } else { 'o' }).collect::<String>());
    }
    lignes.join("\n")
}

let (x_moons, y_moons) = make_moons(300, 0.30, 42);

// SVM a noyau RBF (gamma "scale" calcule explicitement) contre noyau lineaire.
let gamma_moons = gamma_scale(&x_moons);
let params_lin = P::default().with_c(1.0).with_kernel(Kernels::linear()).with_seed(Some(42));
let params_rbf = P::default().with_c(1.0).with_kernel(Kernels::rbf().with_gamma(gamma_moons)).with_seed(Some(42));
let (acc_lin, acc_rbf, mal_classes, carte_lin, carte_rbf) = {
    let svm_lin = SVC::fit(&x_moons, &y_moons, &params_lin).unwrap();
    let svm_rbf = SVC::fit(&x_moons, &y_moons, &params_rbf).unwrap();
    let acc_lin = precision_f64(&y_moons, &svm_lin.predict(&x_moons).unwrap());
    let pred_rbf = svm_rbf.predict(&x_moons).unwrap();
    let acc_rbf = precision_f64(&y_moons, &pred_rbf);
    // smartcore n'expose pas les vecteurs supports : on compte les points mal classes.
    let mal_classes = y_moons.iter().zip(&pred_rbf).filter(|(v, p)| **v as f64 != **p).count();
    let carte_lin = carte_decision(
        |grille| svm_lin.predict(grille).unwrap().into_iter().map(|v| v as u32).collect(),
        &x_moons,
        64,
        24,
    );
    let carte_rbf = carte_decision(
        |grille| svm_rbf.predict(grille).unwrap().into_iter().map(|v| v as u32).collect(),
        &x_moons,
        64,
        24,
    );
    (acc_lin, acc_rbf, mal_classes, carte_lin, carte_rbf)
};

println!("Carte : SVM lineaire -- acc = {:.3}", acc_lin);
println!("{}", carte_lin);
println!();
println!("Carte : SVM RBF -- acc = {:.3}", acc_rbf);
println!("{}", carte_rbf);
println!();
println!("SVM lineaire -- accuracy : {:.3}", acc_lin);
println!("SVM RBF      -- accuracy : {:.3}", acc_rbf);
println!("Points mal classes a l'entrainement (RBF) : {} sur {} points d'entrainement", mal_classes, y_moons.len());
// Premiere carte : une droite coupe les demi-lunes (echec inevitable).
// Seconde carte : une frontiere COURBE epouse les deux classes.

Carte : SVM lineaire -- acc = 0.860


................................................................


................................................................


................................................................


................................................................


................................................................


... (43 outputs left out)

.............ooooooooooooooooooooooooooooooooooooooooooooooooooo


SVM lineaire -- accuracy : 0.860


SVM RBF      -- accuracy : 0.930


Points mal classes a l'entrainement (RBF) : 21 sur 300 points d'entrainement


### Interprétation : le kernel trick rendu visible

**Sortie obtenue** : deux cartes, l'une sous l'autre. Dans la première, le noyau linéaire trace une droite qui coupe les demi-lunes (accuracy modeste). Dans la seconde, le noyau RBF trace une frontière courbe qui suit le creux entre les deux demi-lunes (accuracy supérieure : 0.930 contre 0.860 pour le noyau linéaire) ; 21 points sur 300 restent mal classés à l'entraînement (compte affiché en fin de cellule : smartcore ne donne pas accès aux vecteurs supports).

| Aspect | Noyau linéaire | Noyau RBF |
|--------|----------------|-----------|
| Forme de la frontière | Droite | Courbe, épouse les classes |
| Sépare les demi-lunes ? | Non | Oui |
| Nombre de paramètres appris | Fixe (un vecteur) | Dépend des vecteurs supports |
| Accuracy sur `make_moons` (mesurée dans ce carnet) | 0.860 | 0.930 |

**Points clés** :

1. Le noyau RBF **n'ajoute pas de non-linéarité au modèle** : il ajoute des *dimensions* à l'espace des données. La frontière reste un hyperplan — dans l'espace transformé.
2. La frontière ne dépend **que** des vecteurs supports : déplacer ou supprimer les autres points ne change rien. C'est ce qui fait du SVM un modèle non paramétrique et parcimonieux.
3. Deux hyperparamètres contrôlent la flexibilité : `C` (pénalisation des erreurs, donc largeur de la marge) et `gamma` (rayon d'influence de chaque vecteur support). On étudie `C` à l'exercice 1.

### Exercice 1 : l'effet du paramètre C sur la marge

Le paramètre `C` pénalise les erreurs de classification : un **petit `C`** autorise une marge **large** et plus d'erreurs (modèle plus rigide : la plupart des points tombent dans la marge ou sont mal classés, et deviennent vecteurs supports), un **grand `C`** resserre la marge et tolère moins d'erreurs (modèle plus flexible : seule une fraction des points reste vecteurs supports, ceux qui bordent la marge, y pénètrent ou sont mal classés).

**Objectif** : pour `C` dans `[0.01, 1.0, 100.0]`, ajuster un SVM à noyau RBF (`with_c(c)`, `gamma` de `"scale"` fourni par `gamma_moons`, graine 42) sur `x_moons`/`y_moons`, relever son **accuracy d'entraînement** dans `acc_par_c` (un tableau de trois `f64`, dans l'ordre des valeurs de `C`), puis afficher les trois cartes de décision. La source compte ici les vecteurs supports (`len(modele.support_vectors_)`) ; `SVC` de smartcore 0.7.0 n'en expose aucun : l'accuracy d'entraînement est le témoin mesurable du même phénomène.

**Indice** : une boucle `for c in valeurs_c`, un ajustement par passage, et `carte_decision(...)` pour chaque carte. Deux contraintes de smartcore à respecter, déjà mises en œuvre aux sections 2 et 3 : les paramètres se lient à une variable **avant** l'appel de `SVC::fit` (qui les emprunte), et le modèle ajusté emprunte ses données — il se range donc dans un bloc, dont seuls l'accuracy et la carte ressortent.

In [6]:
// Exercice 1 : effet du parametre C sur l'accuracy d'entrainement (SVM RBF)
use smartcore::linalg::basic::matrix::DenseMatrix;
use smartcore::linalg::basic::arrays::Array;
use smartcore::svm::{Kernels, svc::{SVC, SVCParameters}};

type P = SVCParameters<f64, u32, DenseMatrix<f64>, Vec<u32>>;

// Bruit : melange de Fisher-Yates portant sur les points ET les etiquettes (meme
// permutation), puis bruit gaussien. Generateur xorshift64 : la graine suffit a
// reproduire le jeu de donnees.
fn bruit(x: &mut Vec<Vec<f64>>, y: &mut Vec<u32>, ecart: f64, graine: u64) {
    let mut etat = graine | 1;
    let mut uniform = || {
        etat ^= etat << 13;
        etat ^= etat >> 7;
        etat ^= etat << 17;
        (etat >> 11) as f64 / (1u64 << 53) as f64
    };
    let n = x.len();
    for i in (1..n).rev() {
        let j = (uniform() * (i + 1) as f64) as usize;
        x.swap(i, j);
        y.swap(i, j);
    }
    let mut gaussien = || {
        let u = uniform().max(1e-12);
        (-2.0 * u.ln()).sqrt() * (2.0 * std::f64::consts::PI * uniform()).cos()
    };
    for p in x.iter_mut() {
        p[0] += ecart * gaussien();
        p[1] += ecart * gaussien();
    }
}

// make_moons maison : deux demi-cercles de rayon 1 (exterieur en haut, interieur
// en bas), meme effectif par classe, puis bruit (ecart, graine).
fn make_moons(n: usize, ecart: f64, graine: u64) -> (DenseMatrix<f64>, Vec<u32>) {
    let n_ext = n / 2;
    let n_int = n - n_ext;
    let mut x: Vec<Vec<f64>> = Vec::with_capacity(n);
    let mut y: Vec<u32> = Vec::with_capacity(n);
    for i in 0..n_ext {
        let t = std::f64::consts::PI * i as f64 / (n_ext - 1) as f64;
        x.push(vec![t.cos(), t.sin()]);
        y.push(0);
    }
    for i in 0..n_int {
        let t = std::f64::consts::PI * i as f64 / (n_int - 1) as f64;
        x.push(vec![1.0 - t.cos(), 0.5 - t.sin()]);
        y.push(1);
    }
    bruit(&mut x, &mut y, ecart, graine);
    (DenseMatrix::from_2d_vec(&x).unwrap(), y)
}

// gamma = "scale" de la source : 1 / (nombre de variables x variance de X), la
// variance etant populationnelle et porte sur toutes les valeurs de X.
// smartcore ne fournit pas cette valeur par defaut : elle se calcule.
fn gamma_scale(m: &DenseMatrix<f64>) -> f64 {
    let v: Vec<f64> = m.iter().cloned().collect();
    let n = v.len() as f64;
    let mu: f64 = v.iter().sum::<f64>() / n;
    let variance: f64 = v.iter().map(|z| (z - mu) * (z - mu)).sum::<f64>() / n;
    1.0 / (m.shape().1 as f64 * variance)
}

// Accuracy : fraction des etiquettes bien predites. SVC::predict rend un f64 par
// point (la classe 0.0 ou 1.0), d'ou la comparaison convertie.
fn precision_f64(y: &[u32], p: &[f64]) -> f64 {
    y.iter().zip(p).filter(|(a, b)| **a as f64 == **b).count() as f64 / y.len() as f64
}

// Bornes du plan : etendue des donnees, elargie de 0.5 de chaque cote (comme la source).
fn bornes(x: &DenseMatrix<f64>) -> (f64, f64, f64, f64) {
    let mut x0 = f64::INFINITY;
    let mut x1 = f64::NEG_INFINITY;
    let mut y0 = f64::INFINITY;
    let mut y1 = f64::NEG_INFINITY;
    for i in 0..x.shape().0 {
        let (px, py) = (*x.get((i, 0)), *x.get((i, 1)));
        x0 = x0.min(px);
        x1 = x1.max(px);
        y0 = y0.min(py);
        y1 = y1.max(py);
    }
    (x0 - 0.5, x1 + 0.5, y0 - 0.5, y1 + 0.5)
}

// Carte ASCII de la frontiere : la grille ncol x nlig du plan est classee par le
// modele deja ajuste ; '.' = classe 0, 'o' = classe 1. Premiere ligne = haut du plan.
fn carte_decision(pred: impl Fn(&DenseMatrix<f64>) -> Vec<u32>, x: &DenseMatrix<f64>, ncol: usize, nlig: usize) -> String {
    let (x0, x1, y0, y1) = bornes(x);
    let mut lignes = Vec::with_capacity(nlig);
    for r in (0..nlig).rev() {
        let yy = y0 + (y1 - y0) * (r as f64 + 0.5) / nlig as f64;
        let grille: Vec<Vec<f64>> = (0..ncol)
            .map(|c| vec![x0 + (x1 - x0) * (c as f64 + 0.5) / ncol as f64, yy])
            .collect();
        let classes = pred(&DenseMatrix::from_2d_vec(&grille).unwrap());
        lignes.push(classes.iter().map(|&c| if c == 0 { '.' } else { 'o' }).collect::<String>());
    }
    lignes.join("\n")
}

// Meme jeu de donnees et meme gamma "scale" qu'en section 3.
let (x_moons, y_moons) = make_moons(300, 0.30, 42);
let gamma_moons = gamma_scale(&x_moons);
let valeurs_c = [0.01, 1.0, 100.0];

// TODO etudiant : pour chaque c dans valeurs_c,
//   - lier les parametres a une variable AVANT l'ajustement (SVC::fit les emprunte) :
//       let params = P::default().with_c(c).with_kernel(Kernels::rbf().with_gamma(gamma_moons)).with_seed(Some(42));
//   - ajuster dans un bloc (un SVC ajuste emprunte ses donnees) :
//       let svm = SVC::fit(&x_moons, &y_moons, &params).unwrap();
//   - mesurer precision_f64(&y_moons, &svm.predict(&x_moons).unwrap()) et le ranger dans acc_par_c ;
//   - afficher la carte :
//       println!("{}", carte_decision(|grille| svm.predict(grille).unwrap().into_iter().map(|v| v as u32).collect(), &x_moons, 64, 24));
let acc_par_c: Option<[f64; 3]> = None;  // TODO etudiant : remplacer (accuracy d'entrainement pour C = 0.01, 1.0, 100.0)

println!("Exercice 1 a completer : accuracies par C = {:?}", acc_par_c);

Exercice 1 a completer : accuracies par C = None


## 4. k plus proches voisins : apprendre en mémorisant

Le **k-NN** (k *nearest neighbors*) est le modèle non paramétrique le plus direct : **il n'y a pas de phase d'entraînement**. On stocke toutes les données ; pour prédire la classe d'un nouveau point, on regarde ses `k` voisins les plus proches (au sens d'une distance, par défaut euclidienne) et on prend la **classe majoritaire**. Le seul hyperparamètre est `k`.

Deux extrêmes à éviter :

- `k = 1` : chaque point est sa propre classe → **surajustement**, la frontière est très fracturée (chaque point d'entraînement crée une enclave).
- `k` très grand : le vote lisse tout → **sous-ajustement**, la frontière dégénère vers la classe majoritaire.

> **Référence.** Cover, T. & Hart, P. (1967), *Nearest Neighbor Pattern Classification*, IEEE Transactions on Information Theory 13(1):21-27. La règle du 1-plus-proche-voisin possède une erreur asymptotique bornée par le double de l'erreur de Bayes : un classifieur « paresseux » étonnamment puissant, qui justifie la famille k-NN.

In [7]:
// k-NN sur les demi-lunes : effet du nombre de voisins k
use smartcore::linalg::basic::matrix::DenseMatrix;
use smartcore::linalg::basic::arrays::Array;
use smartcore::neighbors::knn_classifier::{KNNClassifier, KNNClassifierParameters};

// Bruit : melange de Fisher-Yates portant sur les points ET les etiquettes (meme
// permutation), puis bruit gaussien. Generateur xorshift64 : la graine suffit a
// reproduire le jeu de donnees.
fn bruit(x: &mut Vec<Vec<f64>>, y: &mut Vec<u32>, ecart: f64, graine: u64) {
    let mut etat = graine | 1;
    let mut uniform = || {
        etat ^= etat << 13;
        etat ^= etat >> 7;
        etat ^= etat << 17;
        (etat >> 11) as f64 / (1u64 << 53) as f64
    };
    let n = x.len();
    for i in (1..n).rev() {
        let j = (uniform() * (i + 1) as f64) as usize;
        x.swap(i, j);
        y.swap(i, j);
    }
    let mut gaussien = || {
        let u = uniform().max(1e-12);
        (-2.0 * u.ln()).sqrt() * (2.0 * std::f64::consts::PI * uniform()).cos()
    };
    for p in x.iter_mut() {
        p[0] += ecart * gaussien();
        p[1] += ecart * gaussien();
    }
}

// make_moons maison : deux demi-cercles de rayon 1 (exterieur en haut, interieur
// en bas), meme effectif par classe, puis bruit (ecart, graine).
fn make_moons(n: usize, ecart: f64, graine: u64) -> (DenseMatrix<f64>, Vec<u32>) {
    let n_ext = n / 2;
    let n_int = n - n_ext;
    let mut x: Vec<Vec<f64>> = Vec::with_capacity(n);
    let mut y: Vec<u32> = Vec::with_capacity(n);
    for i in 0..n_ext {
        let t = std::f64::consts::PI * i as f64 / (n_ext - 1) as f64;
        x.push(vec![t.cos(), t.sin()]);
        y.push(0);
    }
    for i in 0..n_int {
        let t = std::f64::consts::PI * i as f64 / (n_int - 1) as f64;
        x.push(vec![1.0 - t.cos(), 0.5 - t.sin()]);
        y.push(1);
    }
    bruit(&mut x, &mut y, ecart, graine);
    (DenseMatrix::from_2d_vec(&x).unwrap(), y)
}

// Accuracy : fraction des etiquettes bien predites. KNNClassifier::predict rend
// directement les etiquettes.
fn precision_u32(y: &[u32], p: &[u32]) -> f64 {
    y.iter().zip(p).filter(|(a, b)| a == b).count() as f64 / y.len() as f64
}

// Bornes du plan : etendue des donnees, elargie de 0.5 de chaque cote (comme la source).
fn bornes(x: &DenseMatrix<f64>) -> (f64, f64, f64, f64) {
    let mut x0 = f64::INFINITY;
    let mut x1 = f64::NEG_INFINITY;
    let mut y0 = f64::INFINITY;
    let mut y1 = f64::NEG_INFINITY;
    for i in 0..x.shape().0 {
        let (px, py) = (*x.get((i, 0)), *x.get((i, 1)));
        x0 = x0.min(px);
        x1 = x1.max(px);
        y0 = y0.min(py);
        y1 = y1.max(py);
    }
    (x0 - 0.5, x1 + 0.5, y0 - 0.5, y1 + 0.5)
}

// Carte ASCII de la frontiere : la grille ncol x nlig du plan est classee par le
// modele deja ajuste ; '.' = classe 0, 'o' = classe 1. Premiere ligne = haut du plan.
fn carte_decision(pred: impl Fn(&DenseMatrix<f64>) -> Vec<u32>, x: &DenseMatrix<f64>, ncol: usize, nlig: usize) -> String {
    let (x0, x1, y0, y1) = bornes(x);
    let mut lignes = Vec::with_capacity(nlig);
    for r in (0..nlig).rev() {
        let yy = y0 + (y1 - y0) * (r as f64 + 0.5) / nlig as f64;
        let grille: Vec<Vec<f64>> = (0..ncol)
            .map(|c| vec![x0 + (x1 - x0) * (c as f64 + 0.5) / ncol as f64, yy])
            .collect();
        let classes = pred(&DenseMatrix::from_2d_vec(&grille).unwrap());
        lignes.push(classes.iter().map(|&c| if c == 0 { '.' } else { 'o' }).collect::<String>());
    }
    lignes.join("\n")
}

let (x_moons, y_moons) = make_moons(300, 0.30, 42);
let valeurs_k = [1usize, 15, 75];

// Un ajustement par valeur de k ; l'accuracy se mesure sur les points d'entrainement.
let mut accs_knn: Vec<(usize, f64)> = Vec::new();
let mut cartes_knn: Vec<String> = Vec::new();
for k in valeurs_k {
    let knn = KNNClassifier::fit(&x_moons, &y_moons, KNNClassifierParameters::default().with_k(k)).unwrap();
    accs_knn.push((k, precision_u32(&y_moons, &knn.predict(&x_moons).unwrap())));
    cartes_knn.push(carte_decision(|grille| knn.predict(grille).unwrap(), &x_moons, 64, 24));
}
for ((k, acc), carte) in accs_knn.iter().zip(&cartes_knn) {
    println!("Carte : k={} -- acc = {:.3}", k, acc);
    println!("{}", carte);
    println!();
}
for (k, acc) in &accs_knn {
    println!("k-NN (k={:2}) -- accuracy : {:.3}", k, acc);
}
println!("k=1 : frontiere fracturee (surajustement). k=75 : frontiere trop lisse (sous-ajustement). k=15 offre un bon compromis.");

Carte : k=1 -- acc = 1.000


...............................................ooooooooooooooooo


..............................................oooooooooooooooooo


.............................................ooooooooooooooooooo


............................................oooooooooooooooooooo


...........................................ooooooooooooooooooooo


... (69 outputs left out)

k-NN (k= 1) -- accuracy : 1.000


k-NN (k=15) -- accuracy : 0.913


k-NN (k=75) -- accuracy : 0.890


k=1 : frontiere fracturee (surajustement). k=75 : frontiere trop lisse (sous-ajustement). k=15 offre un bon compromis.


## 5. Pourquoi standardiser est indispensable (surtout pour k-NN)

Le k-NN et le SVM à noyau RBF reposent sur des **distances euclidiennes**. Si une variable a une échelle 100 fois plus grande qu'une autre (par exemple un salaire en euros vs un âge en années), elle **écrase** la distance : l'algorithme « voit » presque uniquement cette variable. La **standardisation** (`StandardScaler` : centrer/réduire, chaque variable de moyenne 0 et d'écart-type 1) remet toutes les variables à la même échelle.

Pour le démontrer, nous **déformons** les demi-lunes en multipliant la première coordonnée par 100 (une variable devient dominante), puis nous comparons un k-NN entraîné sur les variables brutes déformées au **même** k-NN entraîné sur les variables standardisées.

In [8]:
// Demonstration : la standardisation est cruciale pour les modeles bases sur les distances
// On deforme les demi-lunes : la 1re coordonnee est mise a l'echelle x100
use smartcore::linalg::basic::matrix::DenseMatrix;
use smartcore::linalg::basic::arrays::Array;
use smartcore::neighbors::knn_classifier::{KNNClassifier, KNNClassifierParameters};
use smartcore::preprocessing::numerical::{StandardScaler, StandardScalerParameters};
// Les methodes fit et transform du scaler viennent de deux traits de la facade api.
use smartcore::api::{Transformer, UnsupervisedEstimator};

// Bruit : melange de Fisher-Yates portant sur les points ET les etiquettes (meme
// permutation), puis bruit gaussien. Generateur xorshift64 : la graine suffit a
// reproduire le jeu de donnees.
fn bruit(x: &mut Vec<Vec<f64>>, y: &mut Vec<u32>, ecart: f64, graine: u64) {
    let mut etat = graine | 1;
    let mut uniform = || {
        etat ^= etat << 13;
        etat ^= etat >> 7;
        etat ^= etat << 17;
        (etat >> 11) as f64 / (1u64 << 53) as f64
    };
    let n = x.len();
    for i in (1..n).rev() {
        let j = (uniform() * (i + 1) as f64) as usize;
        x.swap(i, j);
        y.swap(i, j);
    }
    let mut gaussien = || {
        let u = uniform().max(1e-12);
        (-2.0 * u.ln()).sqrt() * (2.0 * std::f64::consts::PI * uniform()).cos()
    };
    for p in x.iter_mut() {
        p[0] += ecart * gaussien();
        p[1] += ecart * gaussien();
    }
}

// make_moons maison : deux demi-cercles de rayon 1 (exterieur en haut, interieur
// en bas), meme effectif par classe, puis bruit (ecart, graine).
fn make_moons(n: usize, ecart: f64, graine: u64) -> (DenseMatrix<f64>, Vec<u32>) {
    let n_ext = n / 2;
    let n_int = n - n_ext;
    let mut x: Vec<Vec<f64>> = Vec::with_capacity(n);
    let mut y: Vec<u32> = Vec::with_capacity(n);
    for i in 0..n_ext {
        let t = std::f64::consts::PI * i as f64 / (n_ext - 1) as f64;
        x.push(vec![t.cos(), t.sin()]);
        y.push(0);
    }
    for i in 0..n_int {
        let t = std::f64::consts::PI * i as f64 / (n_int - 1) as f64;
        x.push(vec![1.0 - t.cos(), 0.5 - t.sin()]);
        y.push(1);
    }
    bruit(&mut x, &mut y, ecart, graine);
    (DenseMatrix::from_2d_vec(&x).unwrap(), y)
}

// Accuracy : fraction des etiquettes bien predites. KNNClassifier::predict rend
// directement les etiquettes.
fn precision_u32(y: &[u32], p: &[u32]) -> f64 {
    y.iter().zip(p).filter(|(a, b)| a == b).count() as f64 / y.len() as f64
}

// Bornes du plan : etendue des donnees, elargie de 0.5 de chaque cote (comme la source).
fn bornes(x: &DenseMatrix<f64>) -> (f64, f64, f64, f64) {
    let mut x0 = f64::INFINITY;
    let mut x1 = f64::NEG_INFINITY;
    let mut y0 = f64::INFINITY;
    let mut y1 = f64::NEG_INFINITY;
    for i in 0..x.shape().0 {
        let (px, py) = (*x.get((i, 0)), *x.get((i, 1)));
        x0 = x0.min(px);
        x1 = x1.max(px);
        y0 = y0.min(py);
        y1 = y1.max(py);
    }
    (x0 - 0.5, x1 + 0.5, y0 - 0.5, y1 + 0.5)
}

// Carte ASCII de la frontiere : la grille ncol x nlig du plan est classee par le
// modele deja ajuste ; '.' = classe 0, 'o' = classe 1. Premiere ligne = haut du plan.
fn carte_decision(pred: impl Fn(&DenseMatrix<f64>) -> Vec<u32>, x: &DenseMatrix<f64>, ncol: usize, nlig: usize) -> String {
    let (x0, x1, y0, y1) = bornes(x);
    let mut lignes = Vec::with_capacity(nlig);
    for r in (0..nlig).rev() {
        let yy = y0 + (y1 - y0) * (r as f64 + 0.5) / nlig as f64;
        let grille: Vec<Vec<f64>> = (0..ncol)
            .map(|c| vec![x0 + (x1 - x0) * (c as f64 + 0.5) / ncol as f64, yy])
            .collect();
        let classes = pred(&DenseMatrix::from_2d_vec(&grille).unwrap());
        lignes.push(classes.iter().map(|&c| if c == 0 { '.' } else { 'o' }).collect::<String>());
    }
    lignes.join("\n")
}

let (x_moons, y_moons) = make_moons(300, 0.30, 42);

// On deforme les demi-lunes : la 1re coordonnee est mise a l'echelle x100
let lignes: Vec<Vec<f64>> = (0..x_moons.shape().0)
    .map(|i| vec![*x_moons.get((i, 0)) * 100.0, *x_moons.get((i, 1))])
    .collect();
let x_distordu = DenseMatrix::from_2d_vec(&lignes).unwrap();

// Standardisation (centrage + reduction) pour ramener les 2 variables a la meme echelle
let scaler = StandardScaler::<f64>::fit(&x_distordu, StandardScalerParameters::default()).unwrap();
let x_scaled = scaler.transform(&x_distordu).unwrap();

// Meme k-NN (k=15) sur les donnees brutes deformees vs standardisees
let k = 15;
let knn_brut = KNNClassifier::fit(&x_distordu, &y_moons, KNNClassifierParameters::default().with_k(k)).unwrap();
let knn_std = KNNClassifier::fit(&x_scaled, &y_moons, KNNClassifierParameters::default().with_k(k)).unwrap();
let acc_knn_raw = precision_u32(&y_moons, &knn_brut.predict(&x_distordu).unwrap());
let acc_knn_scaled = precision_u32(&y_moons, &knn_std.predict(&x_scaled).unwrap());

println!("Carte : k-NN brut (x1 x100) -- acc = {:.3}", acc_knn_raw);
println!("{}", carte_decision(|grille| knn_brut.predict(grille).unwrap(), &x_distordu, 64, 24));
println!();
println!("Carte : k-NN standardise -- acc = {:.3}", acc_knn_scaled);
println!("{}", carte_decision(|grille| knn_std.predict(grille).unwrap(), &x_scaled, 64, 24));
println!();
println!("k-NN sur variables brutes deformees  : {:.3}", acc_knn_raw);
println!("k-NN sur variables standardisees     : {:.3}", acc_knn_scaled);
println!("Sans standardisation, la variable x1 (x100) ecrase la distance et la frontiere degenere. La standardisation restitue le bon comportement.");

Carte : k-NN brut (x1 x100) -- acc = 0.773


.....................oo...ooooo..o......oooooooooooooooooooooooo


.....................oo...ooooo..o......oooooooooooooooooooooooo


.....................oo...ooooo..o......oooooooooooooooooooooooo


.....................oo...ooooo..o......oooooooooooooooooooooooo


.....................oo...ooooo..o......oooooooooooooooooooooooo


... (43 outputs left out)

oooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooo


k-NN sur variables brutes deformees  : 0.773


k-NN sur variables standardisees     : 0.920


Sans standardisation, la variable x1 (x100) ecrase la distance et la frontiere degenere. La standardisation restitue le bon comportement.


### Exercice 2 : choisir `k` par séparation train/test

Plutôt que de choisir `k` au hasard, on balaie plusieurs valeurs et on retient celle qui maximise l'accuracy **sur un ensemble de test** (approche par validation simple).

**Objectif** : pour `k` allant de 1 à 49 (impairs), entraîner un `KNNClassifier` (`KNNClassifierParameters::default().with_k(k)`) sur `x_train_k`/`y_train_k` (déjà fournis), mesurer l'accuracy sur `x_test_k`/`y_test_k`, puis identifier le `k` optimal.

**Indice** : stockez les accuracies dans un `Vec<f64>` `acc_par_k` (même longueur que `valeurs_k`), puis prenez l'indice du maximum pour obtenir le meilleur `k` (`valeurs_k[indice]`). En cas d'égalité, la source (`np.argmax`) retient le plus petit `k` : faites de même.

> **Côté Rust.** La séparation fournie est celle de smartcore (`train_test_split`), qui ne stratifie pas ; la source Python passe `stratify=y_moons`.

In [9]:
// Exercice 2 : choisir le meilleur k par separation train/test
use smartcore::linalg::basic::matrix::DenseMatrix;
use smartcore::linalg::basic::arrays::Array;
use smartcore::neighbors::knn_classifier::{KNNClassifier, KNNClassifierParameters};
use smartcore::model_selection::train_test_split;

// Bruit : melange de Fisher-Yates portant sur les points ET les etiquettes (meme
// permutation), puis bruit gaussien. Generateur xorshift64 : la graine suffit a
// reproduire le jeu de donnees.
fn bruit(x: &mut Vec<Vec<f64>>, y: &mut Vec<u32>, ecart: f64, graine: u64) {
    let mut etat = graine | 1;
    let mut uniform = || {
        etat ^= etat << 13;
        etat ^= etat >> 7;
        etat ^= etat << 17;
        (etat >> 11) as f64 / (1u64 << 53) as f64
    };
    let n = x.len();
    for i in (1..n).rev() {
        let j = (uniform() * (i + 1) as f64) as usize;
        x.swap(i, j);
        y.swap(i, j);
    }
    let mut gaussien = || {
        let u = uniform().max(1e-12);
        (-2.0 * u.ln()).sqrt() * (2.0 * std::f64::consts::PI * uniform()).cos()
    };
    for p in x.iter_mut() {
        p[0] += ecart * gaussien();
        p[1] += ecart * gaussien();
    }
}

// make_moons maison : deux demi-cercles de rayon 1 (exterieur en haut, interieur
// en bas), meme effectif par classe, puis bruit (ecart, graine).
fn make_moons(n: usize, ecart: f64, graine: u64) -> (DenseMatrix<f64>, Vec<u32>) {
    let n_ext = n / 2;
    let n_int = n - n_ext;
    let mut x: Vec<Vec<f64>> = Vec::with_capacity(n);
    let mut y: Vec<u32> = Vec::with_capacity(n);
    for i in 0..n_ext {
        let t = std::f64::consts::PI * i as f64 / (n_ext - 1) as f64;
        x.push(vec![t.cos(), t.sin()]);
        y.push(0);
    }
    for i in 0..n_int {
        let t = std::f64::consts::PI * i as f64 / (n_int - 1) as f64;
        x.push(vec![1.0 - t.cos(), 0.5 - t.sin()]);
        y.push(1);
    }
    bruit(&mut x, &mut y, ecart, graine);
    (DenseMatrix::from_2d_vec(&x).unwrap(), y)
}

// Accuracy : fraction des etiquettes bien predites. KNNClassifier::predict rend
// directement les etiquettes.
fn precision_u32(y: &[u32], p: &[u32]) -> f64 {
    y.iter().zip(p).filter(|(a, b)| a == b).count() as f64 / y.len() as f64
}

// k impairs de 1 a 49
let valeurs_k: Vec<usize> = (1..50).step_by(2).collect();

// Separation train/test : 30 % de test, graine 42 (smartcore ne stratifie pas)
let (x_moons, y_moons) = make_moons(300, 0.30, 42);
let (x_train_k, x_test_k, y_train_k, y_test_k) = train_test_split(&x_moons, &y_moons, 0.3, true, Some(42));
println!("Entrainement : {} points   Test : {} points", x_train_k.shape().0, x_test_k.shape().0);

// TODO etudiant : pour chaque k dans valeurs_k, ajuster
//   KNNClassifier::fit(&x_train_k, &y_train_k, KNNClassifierParameters::default().with_k(k))
// puis mesurer precision_u32(&y_test_k, &modele.predict(&x_test_k).unwrap()) et
// remplir acc_par_k (un Vec<f64>, meme longueur que valeurs_k).
let acc_par_k: Option<Vec<f64>> = None;  // TODO etudiant : remplacer (liste des accuracies de test)

// TODO etudiant : meilleur k = valeurs_k[indice du maximum de acc_par_k]
let meilleur_k: Option<usize> = None;  // TODO etudiant : remplacer

println!("Exercice 2 a completer : meilleur k = {:?}", meilleur_k);

Entrainement : 210 points   Test : 90 points


Exercice 2 a completer : meilleur k = None


## 6. Pipeline complète sur données réelles : cancer du sein

Passons à un jeu de données réel, `load_breast_cancer` (déjà utilisé au notebook 2.4) : 569 observations, 30 variables quantitatives de mesures morphologiques de tumeurs, classe binaire (maligne/bénigne). Ici les 30 variables ont des échelles très différentes — la **standardisation** est obligatoire. Nous comparons un SVM (RBF) et un k-NN, **tous deux précédés d'un `StandardScaler`**, sur une séparation train/test de 30 %.

> **Note technique.** Dans un projet réel on assemblerait ces étapes dans une `sklearn.pipeline.Pipeline` pour éviter toute fuite de données (le scaler serait ajusté uniquement sur le train). Ici nous ajustons explicitement le scaler sur le train puis l'appliquons au test, ce qui est strictement équivalent.

> **Côté Rust.** Le même jeu est livré par smartcore (`smartcore::dataset::breast_cancer` : valeurs en `f32`, converties en `f64` ; étiquettes 0 = malin, 1 = bénin). smartcore n'a pas d'objet pipeline : la séquence standardisation puis classifieur s'écrit explicitement, comme dans la note technique. Sa `train_test_split` ne stratifie pas.

In [10]:
// Pipeline reelle sur le cancer du sein : standardisation + SVM RBF vs k-NN
use smartcore::linalg::basic::matrix::DenseMatrix;
use smartcore::linalg::basic::arrays::Array;
use smartcore::svm::{Kernels, svc::{SVC, SVCParameters}};
use smartcore::neighbors::knn_classifier::{KNNClassifier, KNNClassifierParameters};
use smartcore::preprocessing::numerical::{StandardScaler, StandardScalerParameters};
use smartcore::model_selection::train_test_split;
use smartcore::dataset::breast_cancer;
use smartcore::api::{Transformer, UnsupervisedEstimator};

type P = SVCParameters<f64, u32, DenseMatrix<f64>, Vec<u32>>;

// gamma = "scale" de la source : 1 / (nombre de variables x variance de X), la
// variance etant populationnelle et porte sur toutes les valeurs de X.
// smartcore ne fournit pas cette valeur par defaut : elle se calcule.
fn gamma_scale(m: &DenseMatrix<f64>) -> f64 {
    let v: Vec<f64> = m.iter().cloned().collect();
    let n = v.len() as f64;
    let mu: f64 = v.iter().sum::<f64>() / n;
    let variance: f64 = v.iter().map(|z| (z - mu) * (z - mu)).sum::<f64>() / n;
    1.0 / (m.shape().1 as f64 * variance)
}

// Accuracy : fraction des etiquettes bien predites. SVC::predict rend un f64 par
// point (la classe 0.0 ou 1.0), d'ou la comparaison convertie.
fn precision_f64(y: &[u32], p: &[f64]) -> f64 {
    y.iter().zip(p).filter(|(a, b)| **a as f64 == **b).count() as f64 / y.len() as f64
}

// Accuracy : fraction des etiquettes bien predites. KNNClassifier::predict rend
// directement les etiquettes.
fn precision_u32(y: &[u32], p: &[u32]) -> f64 {
    y.iter().zip(p).filter(|(a, b)| a == b).count() as f64 / y.len() as f64
}

// Jeu de donnees reel : mesures morphologiques de tumeurs, classe binaire
// (0 = malin, 1 = benin). smartcore le livre en f32 : conversion en f64.
let jeu = breast_cancer::load_dataset();
let nf = jeu.num_features as usize;
let lignes: Vec<Vec<f64>> = jeu.data.chunks(nf).map(|r| r.iter().map(|&v| v as f64).collect()).collect();
let x_cancer = DenseMatrix::from_2d_vec(&lignes).unwrap();
let y_cancer: Vec<u32> = jeu.target.clone();
println!("X_cancer : ({}, {})   y_cancer : ({},)", x_cancer.shape().0, x_cancer.shape().1, y_cancer.len());
println!("Classes : 0 (malin) x {}   1 (benin) x {}", y_cancer.iter().filter(|&&v| v == 0).count(), y_cancer.iter().filter(|&&v| v == 1).count());

// Separation train/test : 30 % de test, graine 42 (smartcore ne stratifie pas)
let (x_train_c, x_test_c, y_train_c, y_test_c) = train_test_split(&x_cancer, &y_cancer, 0.3, true, Some(42));
println!("Separation train/test : {} / {}", x_train_c.shape().0, x_test_c.shape().0);

// Standardisation ajustee sur le train SEUL (anti-fuite)
let scaler_c = StandardScaler::<f64>::fit(&x_train_c, StandardScalerParameters::default()).unwrap();
let x_train_c_s = scaler_c.transform(&x_train_c).unwrap();
let x_test_c_s = scaler_c.transform(&x_test_c).unwrap();
let gamma_cancer = gamma_scale(&x_train_c_s);

// SVM a noyau RBF (gamma "scale" du train standardise) : le modele ajuste emprunte
// ses donnees, ajustement et prediction vivent dans un bloc.
let params_cancer = P::default().with_c(1.0).with_kernel(Kernels::rbf().with_gamma(gamma_cancer)).with_seed(Some(42));
let acc_svm_cancer = {
    let svm_cancer = SVC::fit(&x_train_c_s, &y_train_c, &params_cancer).unwrap();
    precision_f64(&y_test_c, &svm_cancer.predict(&x_test_c_s).unwrap())
};

// k-NN (k = 5)
let knn_cancer = KNNClassifier::fit(&x_train_c_s, &y_train_c, KNNClassifierParameters::default().with_k(5)).unwrap();
let acc_knn_cancer = precision_u32(&y_test_c, &knn_cancer.predict(&x_test_c_s).unwrap());

println!("Cancer du sein -- accuracy de test");
println!("  SVM (RBF, standardise) : {:.3}", acc_svm_cancer);
println!("  k-NN (k=5, standardise) : {:.3}", acc_knn_cancer);
println!("Les deux modeles non parametriques standardises atteignent une bonne accuracy ; le SVM RBF est generalement legerement meilleur ici.");

X_cancer : (569, 30)   y_cancer : (569,)


Classes : 0 (malin) x 212   1 (benin) x 357


Separation train/test : 399 / 170


Cancer du sein -- accuracy de test


  SVM (RBF, standardise) : 0.965


  k-NN (k=5, standardise) : 0.953


Les deux modeles non parametriques standardises atteignent une bonne accuracy ; le SVM RBF est generalement legerement meilleur ici.


### Exercice 3 : noyau polynomial vs RBF sur `make_circles`

Le noyau RBF n'est pas le seul noyau non linéaire. Le noyau **polynomial** $K(x, x') = (\gamma \langle x, x'\rangle + r)^d$ peut lui aussi courber la frontière, mais avec un biais géométrique différent. `make_circles` (un point à l'intérieur d'un anneau, deux classes concentriques) est un autre cas non séparable linéairement où la séparation est « radiale ».

**Objectif** : le jeu est fourni (`make_circles(300, 0.15, 0.5, 42)` : bruit 0.15, facteur 0.5) ; ajuster un SVM à noyau polynomial de degré 3 (`Kernels::polynomial().with_degree(3.0)`) et un SVM à noyau RBF, tous deux avec `C = 1.0`, `gamma` de `"scale"` fourni par `gamma_cercles` et graine 42 ; ranger leurs accuracies d'entraînement dans `acc_poly` et `acc_rbf`, puis tracer les deux cartes de décision.

**Indice** : une carte pour `Kernels::polynomial()...`, une pour `Kernels::rbf()...`, comme dans la comparaison de la section 3.

In [11]:
// Exercice 3 : noyau polynomial vs RBF sur make_circles
use smartcore::linalg::basic::matrix::DenseMatrix;
use smartcore::linalg::basic::arrays::Array;
use smartcore::svm::{Kernels, svc::{SVC, SVCParameters}};

type P = SVCParameters<f64, u32, DenseMatrix<f64>, Vec<u32>>;

// Bruit : melange de Fisher-Yates portant sur les points ET les etiquettes (meme
// permutation), puis bruit gaussien. Generateur xorshift64 : la graine suffit a
// reproduire le jeu de donnees.
fn bruit(x: &mut Vec<Vec<f64>>, y: &mut Vec<u32>, ecart: f64, graine: u64) {
    let mut etat = graine | 1;
    let mut uniform = || {
        etat ^= etat << 13;
        etat ^= etat >> 7;
        etat ^= etat << 17;
        (etat >> 11) as f64 / (1u64 << 53) as f64
    };
    let n = x.len();
    for i in (1..n).rev() {
        let j = (uniform() * (i + 1) as f64) as usize;
        x.swap(i, j);
        y.swap(i, j);
    }
    let mut gaussien = || {
        let u = uniform().max(1e-12);
        (-2.0 * u.ln()).sqrt() * (2.0 * std::f64::consts::PI * uniform()).cos()
    };
    for p in x.iter_mut() {
        p[0] += ecart * gaussien();
        p[1] += ecart * gaussien();
    }
}

// make_circles maison : deux cercles concentriques (rayon 1 a l'exterieur, rayon
// egal au facteur a l'interieur), meme effectif par classe, puis bruit (ecart, graine).
fn make_circles(n: usize, ecart: f64, facteur: f64, graine: u64) -> (DenseMatrix<f64>, Vec<u32>) {
    let n_ext = n / 2;
    let n_int = n - n_ext;
    let mut x: Vec<Vec<f64>> = Vec::with_capacity(n);
    let mut y: Vec<u32> = Vec::with_capacity(n);
    for i in 0..n_ext {
        let t = 2.0 * std::f64::consts::PI * i as f64 / n_ext as f64;
        x.push(vec![t.cos(), t.sin()]);
        y.push(0);
    }
    for i in 0..n_int {
        let t = 2.0 * std::f64::consts::PI * i as f64 / n_int as f64;
        x.push(vec![facteur * t.cos(), facteur * t.sin()]);
        y.push(1);
    }
    bruit(&mut x, &mut y, ecart, graine);
    (DenseMatrix::from_2d_vec(&x).unwrap(), y)
}

// gamma = "scale" de la source : 1 / (nombre de variables x variance de X), la
// variance etant populationnelle et porte sur toutes les valeurs de X.
// smartcore ne fournit pas cette valeur par defaut : elle se calcule.
fn gamma_scale(m: &DenseMatrix<f64>) -> f64 {
    let v: Vec<f64> = m.iter().cloned().collect();
    let n = v.len() as f64;
    let mu: f64 = v.iter().sum::<f64>() / n;
    let variance: f64 = v.iter().map(|z| (z - mu) * (z - mu)).sum::<f64>() / n;
    1.0 / (m.shape().1 as f64 * variance)
}

// Accuracy : fraction des etiquettes bien predites. SVC::predict rend un f64 par
// point (la classe 0.0 ou 1.0), d'ou la comparaison convertie.
fn precision_f64(y: &[u32], p: &[f64]) -> f64 {
    y.iter().zip(p).filter(|(a, b)| **a as f64 == **b).count() as f64 / y.len() as f64
}

// Bornes du plan : etendue des donnees, elargie de 0.5 de chaque cote (comme la source).
fn bornes(x: &DenseMatrix<f64>) -> (f64, f64, f64, f64) {
    let mut x0 = f64::INFINITY;
    let mut x1 = f64::NEG_INFINITY;
    let mut y0 = f64::INFINITY;
    let mut y1 = f64::NEG_INFINITY;
    for i in 0..x.shape().0 {
        let (px, py) = (*x.get((i, 0)), *x.get((i, 1)));
        x0 = x0.min(px);
        x1 = x1.max(px);
        y0 = y0.min(py);
        y1 = y1.max(py);
    }
    (x0 - 0.5, x1 + 0.5, y0 - 0.5, y1 + 0.5)
}

// Carte ASCII de la frontiere : la grille ncol x nlig du plan est classee par le
// modele deja ajuste ; '.' = classe 0, 'o' = classe 1. Premiere ligne = haut du plan.
fn carte_decision(pred: impl Fn(&DenseMatrix<f64>) -> Vec<u32>, x: &DenseMatrix<f64>, ncol: usize, nlig: usize) -> String {
    let (x0, x1, y0, y1) = bornes(x);
    let mut lignes = Vec::with_capacity(nlig);
    for r in (0..nlig).rev() {
        let yy = y0 + (y1 - y0) * (r as f64 + 0.5) / nlig as f64;
        let grille: Vec<Vec<f64>> = (0..ncol)
            .map(|c| vec![x0 + (x1 - x0) * (c as f64 + 0.5) / ncol as f64, yy])
            .collect();
        let classes = pred(&DenseMatrix::from_2d_vec(&grille).unwrap());
        lignes.push(classes.iter().map(|&c| if c == 0 { '.' } else { 'o' }).collect::<String>());
    }
    lignes.join("\n")
}

// Jeu fourni : deux cercles concentriques (facteur 0.5), bruit gaussien d'ecart 0.15
let (x_cercles, y_cercles) = make_circles(300, 0.15, 0.5, 42);
let gamma_cercles = gamma_scale(&x_cercles);

// TODO etudiant : pour chaque noyau (polynomial de degre 3, puis RBF), dans un bloc
// (un SVC ajuste emprunte ses donnees) :
//   let params = P::default().with_c(1.0).with_kernel(Kernels::polynomial().with_degree(3.0).with_gamma(gamma_cercles)).with_seed(Some(42));
//   let svm = SVC::fit(&x_cercles, &y_cercles, &params).unwrap();
// puis mesurer precision_f64(&y_cercles, &svm.predict(&x_cercles).unwrap()), l'affecter
// a acc_poly (ou acc_rbf avec Kernels::rbf().with_gamma(gamma_cercles)) et afficher :
//   println!("{}", carte_decision(|grille| svm.predict(grille).unwrap().into_iter().map(|v| v as u32).collect(), &x_cercles, 64, 24));
let acc_poly: Option<f64> = None;  // TODO etudiant : remplacer (accuracy d'entrainement, noyau polynomial de degre 3)
let acc_rbf: Option<f64> = None;  // TODO etudiant : remplacer (accuracy d'entrainement, noyau RBF)

println!("Exercice 3 a completer : comparer la frontiere polynomiale vs RBF sur make_circles");

Exercice 3 a completer : comparer la frontiere polynomiale vs RBF sur make_circles


## Synthèse : non paramétrique supervisé vs le reste du socle

Les **SVM** (noyaux linéaire, RBF, polynomial) et le **k-NN** complètent la famille supervisée du socle ML. Là où la régression logistique (2.3) et les arbres peu profonds tracent des frontières simples, les SVM à noyau et le k-NN suivent des structures **non linéaires complexes** sans supposer de forme paramétrique. Le concept-phare de ce notebook — le **kernel trick** — montre qu'un séparateur linéaire dans un espace transformé produit une frontière courbe dans l'espace d'origine : la capacité du modèle à « courber » n'est pas ajoutée au modèle, elle vient du changement d'espace.

**Repères transverses** :

- Le notebook [2.6](2.6-Clustering-KMeans-PCA.ipynb) traitait du **non supervisé** (KMeans, ACP) : aucun label. Ici, les labels sont **connus** — c'est la distinction supervision/non-supervision, orthogonale au paramétrique/non-paramétrique.
- Comme pour tout modèle basé sur les distances (k-NN, SVM RBF), la **standardisation** est non négociable — c'est la leçon de la section 5.
- SVM et k-NN restent pertinents dans les pipelines agentic qui suivront : ce sont des références solides, interprétables géométriquement, contre lesquelles juger ce qu'un agent LLM produira comme classifieur.

> **Référence.** Vapnik, V. (1998), *Statistical Learning Theory*, Wiley. Cadre théorique des SVM : minimisation structurelle du risque, marge et bornes de généralisation.

Avec ce notebook, la série `02-ML-Cours` couvre désormais l'essentiel des grandes familles : modèles **linéaires** (2.3), **arbres et ensembles** (2.4), **modèles non paramétriques à noyaux et par distance** (2.7), sans oublier l'évaluation rigoureuse (2.5) et le non supervisé (2.6). Deux notebooks théoriques prolongent ensuite la série : [2.8](2.8-Theorie-PAC.ipynb) (théorie PAC, dimension VC) et [2.9](2.9-Grokking-Generalisation.ipynb) (grokking). La suite logique est ensuite l'utilisation de ce socle comme **référent jugeable** des labs agentic (`06-Agents-de-Donnees`, `07-Orchestration-d-Agents`).

## References

1. Cortes, C. & Vapnik, V. (1995). *Support-Vector Networks*. Machine Learning 20(3):273-297. — Les SVM pour la classification : hyperplan à marge maximale, vecteurs supports, introduction des noyaux.
2. Vapnik, V. (1998). *Statistical Learning Theory*. Wiley. — Cadre théorique des SVM : minimisation structurelle du risque, bornes de généralisation fondées sur la marge.
3. Cristianini, N. & Shawe-Taylor, J. (2000). *An Introduction to Support Vector Machines and Other Kernel-based Learning Methods*. Cambridge University Press. — Présentation pédagogique du kernel trick et des noyaux (linéaire, polynomial, RBF).
4. Cover, T. & Hart, P. (1967). *Nearest Neighbor Pattern Classification*. IEEE Transactions on Information Theory 13(1):21-27. — La règle du 1-plus-proche-voisin et sa borne d'erreur asymptotique (≤ 2× l'erreur de Bayes).
5. Hastie, T., Tibshirani, R. & Friedman, J. (2009). *The Elements of Statistical Learning*. Springer (2e éd.), §§12.2-12.3, 13.3. — SVM, noyaux et méthodes basées sur les voisins (k-NN).
6. Pedregosa, F. et al. (2011). *Scikit-learn: Machine Learning in Python*. Journal of Machine Learning Research 12:2825-2830. — `SVC`, `KNeighborsClassifier`, `StandardScaler`, `make_moons`, `make_circles`, `load_breast_cancer`.

*Implémentation Rust* : crate `smartcore` 0.7.0 (licence Apache-2.0) — `SVC`, `KNNClassifier`, `StandardScaler`, `train_test_split`, `breast_cancer` ; `make_moons` et `make_circles` réécrits dans le carnet.

## À retenir

Sur les demi-lunes, le noyau RBF permet une frontière non linéaire ; k-NN s’appuie sur les voisins et dépend fortement de leur nombre et de l’échelle des variables. La standardisation doit être apprise dans le pipeline sur les données d’entraînement.

## Vérifiez votre compréhension

1. Pourquoi le noyau RBF aide-t-il sur les demi-lunes ?

<details><summary>Réponse 1</summary>

Il permet une frontière non linéaire là où une séparation droite coupe les croissants.

</details>

2. Pourquoi standardiser avant k-NN ?

<details><summary>Réponse 2</summary>

Sans changement d’échelle, une variable numériquement dominante peut écraser les autres dans la distance.

</details>

3. Que risque-t-on avec un seul voisin ?

<details><summary>Réponse 3</summary>

Une frontière sensible au bruit et un surajustement ; il faut mesurer sur des données séparées.

</details>

4. Pourquoi placer la standardisation dans un pipeline ?

<details><summary>Réponse 4</summary>

Pour ajuster ses paramètres sur chaque entraînement, sans utiliser les données de validation ou de test.

</details>

## Pour aller plus loin

[SMO from scratch](2.7b-SMO-From-Scratch.ipynb) détaille l’optimisation du SVM à noyau.